# Part F — Bonus: RabbitMQ to FreshCart Snowflake Lakehouse



## 1. Install and start RabbitMQ

This cell installs the local RabbitMQ broker and Python client. It is safe to rerun if the packages are already installed.

In [1]:
!apt-get update -qq
!apt-get install -y rabbitmq-server
!pip -q install pika snowflake-connector-python pandas
!service rabbitmq-server start
!rabbitmqctl status

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  erlang-asn1 erlang-base erlang-crypto erlang-eldap erlang-ftp erlang-inets
  erlang-mnesia erlang-os-mon erlang-parsetools erlang-public-key
  erlang-runtime-tools erlang-snmp erlang-ssl erlang-syntax-tools erlang-tftp
  erlang-tools erlang-xmerl libsctp1 logrotate
Suggested packages:
  erlang erlang-manpages erlang-doc lksctp-tools bsd-mailx | mailx
The following NEW packages will be installed:
  erlang-asn1 erlang-base erlang-crypto erlang-eldap erlang-ftp erlang-inets
  erlang-mnesia erlang-os-mon erlang-parsetools erlang-public-key
  erlang-runtime-tools erlang-snmp erlang-ssl erlang-syntax-tools erlang-tftp
  erlang-tools erlang-xmerl libsctp1 lo

## 2. Connect to the existing FreshCart Snowflake database



In [2]:
from getpass import getpass
from pathlib import Path
from datetime import datetime, timezone
import os, gzip, json, time, uuid

import snowflake.connector
import pandas as pd

SNOWFLAKE_ACCOUNT = "ke60475.ap-southeast-7.aws"
SNOWFLAKE_USER = "VAIDEHEE"
SNOWFLAKE_ROLE = "ACCOUNTADMIN"
SNOWFLAKE_PASSWORD = getpass("Enter your Snowflake password: ")

LANDING_DIR = "/content/freshcart_landing"
Path(LANDING_DIR).mkdir(parents=True, exist_ok=True)

conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT,
    user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD,
    authenticator="snowflake",
    role=SNOWFLAKE_ROLE,
    session_parameters={"QUERY_TAG": "FreshCart_RabbitMQ_Bonus"}
)

def query(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchall()

def show(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        columns = [c[0] for c in cur.description]
    return pd.DataFrame(rows, columns=columns)

query("USE DATABASE FRESHCART")
query("USE WAREHOUSE FRESHCART_WH")
print("Connected:", query("SELECT CURRENT_USER(), CURRENT_DATABASE(), CURRENT_WAREHOUSE()")[0])

Enter your Snowflake password: ··········


/usr/local/lib/python3.13/dist-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


Connected: ('VAIDEHEE', 'FRESHCART', 'FRESHCART_WH')


## 3. Reuse the Part B landing-file and Snowflake load functions


In [3]:
def write_landing(source, records, run_id):
    """Save source records in a compressed JSON Lines file and return its path."""
    folder = Path(LANDING_DIR) / source
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{source}__{run_id}.json.gz"
    now = datetime.now(timezone.utc).isoformat()
    with gzip.open(path, "wt", encoding="utf-8") as f:
        for rec in records:
            envelope = {
                "source": source,
                "run_id": run_id,
                "extracted_at": now,
                "data": rec
            }
            f.write(json.dumps(envelope, ensure_ascii=False) + "\n")
    print(f"{source}: {len(records)} records -> {path}")
    return path

def load_file(local_path, table, stage_dir):
    """PUT a local gzip JSONL file to the Snowflake stage and COPY it into Bronze."""
    local_path = Path(local_path).resolve()
    if not local_path.exists():
        raise FileNotFoundError(f"Landing file not found: {local_path}")

    uri = "file://" + local_path.as_posix()
    query(
        f"PUT '{uri}' @bronze.landing/{stage_dir}/ "
        "AUTO_COMPRESS=FALSE OVERWRITE=FALSE"
    )

    rows = query(f"""
        COPY INTO {table}
            (source, raw, run_id, extracted_at, src_file, src_row)
        FROM (
            SELECT
                $1:source::STRING,
                $1:data,
                $1:run_id::STRING,
                $1:extracted_at::TIMESTAMP_TZ,
                METADATA$FILENAME,
                METADATA$FILE_ROW_NUMBER
            FROM @bronze.landing/{stage_dir}/
        )
        FILE_FORMAT = (FORMAT_NAME = 'bronze.ff_ndjson')
        ON_ERROR = 'ABORT_STATEMENT'
    """)

    loaded = sum(int(r[3]) for r in rows if len(r) > 3)
    print(f"{loaded} rows -> {table}")
    return loaded

print("Part B landing/load functions are ready.")

Part B landing/load functions are ready.


## 4. Define the RabbitMQ order-event dispatcher


In [4]:
import pika

EXCHANGE = "freshcart-orders"
ROUTING_KEY = "freshcart.order.created"

def publish_freshcart_orders(duration_seconds=60, interval_seconds=2):
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host="localhost")
    )
    channel = connection.channel()
    channel.exchange_declare(
        exchange=EXCHANGE,
        exchange_type="topic",
        durable=True
    )

    started = time.time()
    count = 0
    print(f"Publishing FreshCart orders for {duration_seconds} seconds...")

    try:
        while time.time() - started < duration_seconds:
            count += 1
            now = datetime.now(timezone.utc).replace(tzinfo=None)
            record = {
                "order_id": f"STR-{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}-{count:03d}",
                "cust_email": "arjun.k@yahoo.in",
                "cust_name": "Arjun K",
                "order_date": "2026-09-02",
                "amount": f"{250 + (count % 5) * 50:.2f}",
                "currency": "INR",
                "pincode": "560034",
                "updated_at": now.strftime("%Y-%m-%d %H:%M:%S")
            }
            channel.basic_publish(
                exchange=EXCHANGE,
                routing_key=ROUTING_KEY,
                body=json.dumps(record),
                properties=pika.BasicProperties(
                    content_type="application/json",
                    delivery_mode=2
                )
            )
            print(f"Published {record['order_id']} | amount={record['amount']} INR")
            time.sleep(interval_seconds)
    finally:
        connection.close()

    print(f"Dispatcher finished. Published {count} messages.")
    return count

## 5. Define a one-minute RabbitMQ consumer


In [5]:
def collect_freshcart_orders(duration_seconds=60):
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host="localhost")
    )
    channel = connection.channel()
    channel.exchange_declare(
        exchange=EXCHANGE,
        exchange_type="topic",
        durable=True
    )
    result = channel.queue_declare(queue="", exclusive=True)
    queue_name = result.method.queue
    channel.queue_bind(
        exchange=EXCHANGE,
        queue=queue_name,
        routing_key=ROUTING_KEY
    )
    channel.basic_qos(prefetch_count=50)

    collected = []
    deadline = time.time() + duration_seconds

    def callback(ch, method, properties, body):
        try:
            record = json.loads(body.decode("utf-8"))
            required = {
                "order_id", "cust_email", "cust_name", "order_date",
                "amount", "currency", "pincode", "updated_at"
            }
            missing = required - set(record)
            if missing:
                print(f"NACK malformed message; missing={sorted(missing)}")
                ch.basic_nack(
                    delivery_tag=method.delivery_tag,
                    requeue=False
                )
                return

            collected.append(record)
            print(f"Received {record['order_id']} | ACK")
            ch.basic_ack(delivery_tag=method.delivery_tag)
        except Exception as exc:
            print(f"NACK unreadable message: {exc}")
            ch.basic_nack(
                delivery_tag=method.delivery_tag,
                requeue=False
            )

    channel.basic_consume(
        queue=queue_name,
        on_message_callback=callback,
        auto_ack=False
    )

    print(f"Consumer listening for {duration_seconds} seconds...")
    try:
        while time.time() < deadline:
            connection.process_data_events(time_limit=1)
    finally:
        if channel.is_open:
            channel.close()
        connection.close()

    print(f"Consumer collected {len(collected)} order messages.")
    return collected

## 6. Run the dispatcher and collect for one minute


In [6]:
import threading

published_result = {"count": None}
def dispatcher_job():
    published_result["count"] = publish_freshcart_orders(
        duration_seconds=60,
        interval_seconds=2
    )

dispatcher_thread = threading.Thread(target=dispatcher_job, daemon=True)
dispatcher_thread.start()

stream_records = collect_freshcart_orders(duration_seconds=60)
dispatcher_thread.join()

print("\nPublished:", published_result["count"])
print("Collected:", len(stream_records))
assert len(stream_records) > 0, "No messages were collected. Check RabbitMQ and rerun this section."
assert len({r["order_id"] for r in stream_records}) == len(stream_records), "Duplicate stream order IDs detected."

Publishing FreshCart orders for 60 seconds...
Published STR-20261115121657-001 | amount=300.00 INR
Consumer listening for 60 seconds...
Published STR-20261115121659-002 | amount=350.00 INR
Received STR-20261115121659-002 | ACK
Published STR-20261115121701-003 | amount=400.00 INR
Received STR-20261115121701-003 | ACK
Published STR-20261115121703-004 | amount=450.00 INR
Received STR-20261115121703-004 | ACK
Published STR-20261115121705-005 | amount=250.00 INR
Received STR-20261115121705-005 | ACK
Published STR-20261115121707-006 | amount=300.00 INR
Received STR-20261115121707-006 | ACK
Published STR-20261115121709-007 | amount=350.00 INR
Received STR-20261115121709-007 | ACK
Published STR-20261115121711-008 | amount=400.00 INR
Received STR-20261115121711-008 | ACK
Published STR-20261115121713-009 | amount=450.00 INR
Received STR-20261115121713-009 | ACK
Published STR-20261115121715-010 | amount=250.00 INR
Received STR-20261115121715-010 | ACK
Published STR-20261115121717-011 | amount=300

## 7. Write the collected events to a raw landing file and load Bronze


In [7]:
stream_run_id = "stream_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")

stream_file = write_landing(
    source="stream",
    records=stream_records,
    run_id=stream_run_id
)

loaded_stream_rows = load_file(
    stream_file,
    table="bronze.orders_raw",
    stage_dir=f"orders_stream/{stream_run_id}"
)

print("Stream run ID:", stream_run_id)
print("Expected records:", len(stream_records))
print("Loaded Bronze rows:", loaded_stream_rows)
assert loaded_stream_rows == len(stream_records), "Bronze load count does not match collected message count."

stream: 29 records -> /content/freshcart_landing/stream/stream__stream_20261115_121808_734733.json.gz


29 rows -> bronze.orders_raw
Stream run ID: stream_20261115_121808_734733
Expected records: 29
Loaded Bronze rows: 29


## 8. Extend the Part B cleaning SQL to include the stream source


In [9]:
SQL_10_ORDERS_CLEAN_STREAM = r"""
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
    SELECT
        ingest_seq, source,
        raw:order_id::STRING AS order_id,
        raw:cust_email::STRING AS cust_email,
        raw:cust_name::STRING AS cust_name,
        raw:order_date::STRING AS order_date,
        raw:amount::STRING AS amount,
        raw:currency::STRING AS currency,
        raw:pincode::STRING AS pincode,
        raw:updated_at::STRING AS updated_at
    FROM bronze.orders_raw
    WHERE source = 'web'

    UNION ALL

    SELECT
        ingest_seq, source,
        raw:orderId::STRING,
        raw:customer.email::STRING,
        raw:customer.name::STRING,
        raw:orderDate::STRING,
        raw:total.value::STRING,
        raw:total.currency::STRING,
        raw:deliveryPin::STRING,
        raw:updatedAt::STRING
    FROM bronze.orders_raw
    WHERE source = 'mobile'

    UNION ALL

    SELECT
        ingest_seq, source,
        raw:order_id::STRING,
        raw:cust_email::STRING,
        raw:cust_name::STRING,
        raw:order_date::STRING,
        raw:amount::STRING,
        raw:currency::STRING,
        raw:pincode::STRING,
        raw:updated_at::STRING
    FROM bronze.orders_raw
    WHERE source = 'stream'
),
typed AS (
    SELECT
        ingest_seq,
        source,
        TRIM(order_id) AS order_id,
        LOWER(TRIM(cust_email)) AS email,
        INITCAP(RTRIM(TRIM(cust_name), '.')) AS name,
        COALESCE(
            TRY_TO_DATE(order_date, 'YYYY-MM-DD'),
            TRY_TO_DATE(order_date, 'DD/MM/YYYY'),
            TRY_TO_DATE(order_date, 'MON DD, YYYY')
        ) AS order_date,
        TRY_TO_DECIMAL(
            REGEXP_REPLACE(amount, '[^0-9.-]', ''),
            12, 2
        ) AS amount,
        UPPER(TRIM(currency)) AS currency,
        TRIM(pincode) AS pincode,
        TRY_TO_TIMESTAMP_NTZ(updated_at) AS updated_at
    FROM unified
)
SELECT *,
    ARRAY_CONSTRUCT_COMPACT(
        IFF(order_id IS NULL, 'missing_order_id', NULL),
        IFF(
            email IS NULL OR
            NOT REGEXP_LIKE(email, '^[^@[:space:]]+@[^@[:space:]]+[.][a-z]{2,}$'),
            'bad_email', NULL
        ),
        IFF(
            pincode IS NULL OR
            NOT REGEXP_LIKE(pincode, '^[1-9][0-9]{5}$'),
            'bad_pincode', NULL
        ),
        IFF(order_date IS NULL, 'unparseable_date', NULL),
        IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),
        IFF(amount <= 0, 'non_positive_amount', NULL),
        IFF(currency NOT IN ('INR', 'USD'), 'unsupported_currency', NULL)
    ) AS reasons
FROM typed;

CREATE OR REPLACE TABLE silver.orders_clean AS
SELECT
    * EXCLUDE reasons,
    (amount IS NULL) AS amount_missing
FROM silver.orders_checked
WHERE ARRAY_SIZE(reasons) = 0;

INSERT INTO quarantine.orders
    (run_id, ingest_seq, source, order_id, reasons, row_data)
SELECT
    $run_id,
    c.ingest_seq,
    c.source,
    c.order_id,
    c.reasons,
    OBJECT_CONSTRUCT(*)
FROM silver.orders_checked c
WHERE ARRAY_SIZE(c.reasons) > 0
AND NOT EXISTS (
    SELECT 1
    FROM quarantine.orders q
    WHERE q.source = c.source
      AND EQUAL_NULL(q.order_id, c.order_id)
      AND EQUAL_NULL(
          q.row_data:UPDATED_AT::TIMESTAMP_NTZ,
          c.updated_at
      )
)
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.source, c.order_id, c.updated_at
    ORDER BY c.ingest_seq DESC
) = 1;
"""

print("Stream-aware cleaning SQL is ready.")

Stream-aware cleaning SQL is ready.


## 9. Run the FreshCart transformations and publish Gold



In [10]:
# Reuse the downstream SQL definitions from Part B.
# These definitions are included below so this bonus notebook can run independently
# after connecting to the already-prepared Part B Snowflake database.

SQL_20_ORDERS_DEDUP = r"""
CREATE OR REPLACE TABLE silver.orders_dedup AS
SELECT *
FROM silver.orders_clean
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_id
    ORDER BY updated_at DESC, ingest_seq DESC
) = 1;
"""


SQL_30_CUSTOMERS_SCD2 = r"""
CREATE OR REPLACE TABLE dim.customers_scd2 AS
WITH events AS (
    SELECT
        raw:cust_id::STRING AS cust_id,
        LOWER(TRIM(raw:email::STRING)) AS email,
        INITCAP(raw:tier::STRING) AS tier,
        TRY_TO_DATE(raw:effective_from::STRING) AS valid_from,
        loaded_at,
        ingest_seq
    FROM bronze.customers_raw
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY cust_id, valid_from
        ORDER BY loaded_at DESC, ingest_seq DESC
    ) = 1
),
ranged AS (
    SELECT *,
        DATEADD(
            day,
            -1,
            LEAD(valid_from) OVER (
                PARTITION BY cust_id
                ORDER BY valid_from
            )
        ) AS prior_valid_to
    FROM events
)
SELECT
    cust_id,
    email,
    tier,
    valid_from,
    COALESCE(prior_valid_to, '9999-12-31'::DATE) AS valid_to,
    prior_valid_to IS NULL AS is_current
FROM ranged;
"""


SQL_31_FX_DAILY = r"""
CREATE OR REPLACE TABLE ref.fx_daily AS
WITH published AS (
    SELECT
        r.raw:base::STRING AS currency,
        f.key::DATE AS rate_date,
        f.value:INR::NUMBER(12,4) AS rate_to_inr
    FROM bronze.fx_raw r,
         LATERAL FLATTEN(input => r.raw:rates) f
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY currency, rate_date
        ORDER BY r.loaded_at DESC, r.ingest_seq DESC
    ) = 1
),
bounds AS (
    SELECT
        MIN(rate_date) AS d0,
        GREATEST(MAX(rate_date), CURRENT_DATE()) AS d1
    FROM published
),
spine AS (
    SELECT
        DATEADD(
            day,
            ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1,
            b.d0
        ) AS d
    FROM bounds b,
         TABLE(GENERATOR(ROWCOUNT => 4000))
    QUALIFY d <= (SELECT d1 FROM bounds)
),
filled AS (
    SELECT
        c.currency,
        s.d AS rate_date,
        LAST_VALUE(p.rate_to_inr) IGNORE NULLS OVER (
            PARTITION BY c.currency
            ORDER BY s.d
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS rate_to_inr,
        p.rate_to_inr IS NULL AS is_carried_forward
    FROM spine s
    CROSS JOIN (
        SELECT DISTINCT currency
        FROM published
    ) c
    LEFT JOIN published p
        ON p.currency = c.currency
       AND p.rate_date = s.d
)
SELECT
    currency,
    rate_date,
    rate_to_inr,
    is_carried_forward
FROM filled

UNION ALL

SELECT
    'INR',
    d,
    1.0,
    FALSE
FROM spine;
"""


SQL_32_PINCODE_GEO = r"""
MERGE INTO ref.pincode_geo t
USING (
    SELECT
        p.pincode,
        p.city,
        p.state,
        sr.region
    FROM (
        SELECT
            raw:pincode::STRING AS pincode,
            raw:response[0]:PostOffice[0]:District::STRING AS city,
            raw:response[0]:PostOffice[0]:State::STRING AS state,
            loaded_at,
            ingest_seq
        FROM bronze.pincode_raw
        WHERE raw:response[0]:Status::STRING = 'Success'
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY pincode
            ORDER BY loaded_at DESC, ingest_seq DESC
        ) = 1
    ) p
    LEFT JOIN ref.state_region sr
        ON sr.state = p.state
) s
ON t.pincode = s.pincode

WHEN MATCHED AND (
    t.city <> s.city
    OR t.state <> s.state
    OR NVL(t.region, '') <> NVL(s.region, '')
)
THEN UPDATE SET
    city = s.city,
    state = s.state,
    region = s.region,
    updated_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
    pincode,
    city,
    state,
    region,
    updated_at
)
VALUES (
    s.pincode,
    s.city,
    s.state,
    s.region,
    CURRENT_TIMESTAMP()
);
"""


SQL_40_GOLD = r"""
CREATE OR REPLACE TABLE silver.orders_enriched AS
SELECT
    o.*,
    c.cust_id,
    c.tier,
    m.multiplier,
    fx.rate_to_inr AS fx_rate,
    ROUND(o.amount * fx.rate_to_inr, 2) AS amount_inr,
    FLOOR(amount_inr / 100) * m.multiplier AS loyalty_points,
    amount_inr >= 2000 AS is_high_value,
    g.city,
    g.state,
    g.region
FROM silver.orders_dedup o

LEFT JOIN dim.customers_scd2 c
    ON c.email = o.email
   AND o.order_date BETWEEN c.valid_from AND c.valid_to

LEFT JOIN ref.tier_multiplier m
    ON m.tier = c.tier

LEFT JOIN ref.fx_daily fx
    ON fx.currency = o.currency
   AND fx.rate_date = o.order_date

LEFT JOIN ref.pincode_geo g
    ON g.pincode = o.pincode;


MERGE INTO gold.fct_orders t
USING (
    SELECT
        order_id,
        order_date,
        cust_id,
        email,
        tier,
        multiplier,
        amount,
        currency,
        fx_rate,
        amount_inr,
        loyalty_points,
        is_high_value,
        city,
        state,
        region,
        updated_at,
        HASH(
            order_date,
            cust_id,
            email,
            tier,
            multiplier,
            amount,
            currency,
            fx_rate,
            amount_inr,
            loyalty_points,
            is_high_value,
            city,
            state,
            region,
            updated_at
        ) AS row_hash
    FROM silver.orders_enriched
) s
ON t.order_id = s.order_id

WHEN MATCHED AND t.row_hash <> s.row_hash
THEN UPDATE SET
    order_date = s.order_date,
    cust_id = s.cust_id,
    email = s.email,
    tier = s.tier,
    multiplier = s.multiplier,
    amount = s.amount,
    currency = s.currency,
    fx_rate = s.fx_rate,
    amount_inr = s.amount_inr,
    loyalty_points = s.loyalty_points,
    is_high_value = s.is_high_value,
    city = s.city,
    state = s.state,
    region = s.region,
    updated_at = s.updated_at,
    row_hash = s.row_hash,
    published_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
    order_id,
    order_date,
    cust_id,
    email,
    tier,
    multiplier,
    amount,
    currency,
    fx_rate,
    amount_inr,
    loyalty_points,
    is_high_value,
    city,
    state,
    region,
    updated_at,
    row_hash,
    published_at
)
VALUES (
    s.order_id,
    s.order_date,
    s.cust_id,
    s.email,
    s.tier,
    s.multiplier,
    s.amount,
    s.currency,
    s.fx_rate,
    s.amount_inr,
    s.loyalty_points,
    s.is_high_value,
    s.city,
    s.state,
    s.region,
    s.updated_at,
    s.row_hash,
    CURRENT_TIMESTAMP()
);
"""


# Execute the stream-aware transformations in sequence.
def run_stream_transformations():

    query(f"SET run_id = '{stream_run_id}'")

    scripts = [
        SQL_10_ORDERS_CLEAN_STREAM,
        SQL_20_ORDERS_DEDUP,
        SQL_30_CUSTOMERS_SCD2,
        SQL_31_FX_DAILY,
        SQL_32_PINCODE_GEO,
        SQL_40_GOLD
    ]

    for script in scripts:

        results = conn.execute_string(
            script,
            remove_comments=True
        )

        # Consume cursor results so statement execution completes cleanly.
        for cur in results:
            try:
                cur.fetchall()
            except Exception:
                pass

    print("Stream-aware FreshCart transformations completed.")


# Run all transformations.
run_stream_transformations()

Stream-aware FreshCart transformations completed.


## 10. Verify that the streamed orders reached Bronze, Silver and Gold


In [11]:
print("Bronze stream rows:")

display(show(f"""
    SELECT
        source,
        run_id,
        COUNT(*) AS row_count
    FROM bronze.orders_raw
    WHERE source = 'stream'
      AND run_id = '{stream_run_id}'
    GROUP BY source, run_id
"""))


print("Streamed Gold orders:")

display(show(f"""
    SELECT
        order_id,
        email,
        tier,
        amount,
        currency,
        amount_inr,
        loyalty_points,
        city,
        state,
        region
    FROM gold.fct_orders
    WHERE order_id IN (
        SELECT raw:order_id::STRING
        FROM bronze.orders_raw
        WHERE source = 'stream'
          AND run_id = '{stream_run_id}'
    )
    ORDER BY order_id
"""))

Bronze stream rows:


,SOURCE,RUN_ID,ROW_COUNT
0,stream,stream_20261115_121808_734733,29


Streamed Gold orders:


,ORDER_ID,EMAIL,TIER,AMOUNT,CURRENCY,AMOUNT_INR,LOYALTY_POINTS,CITY,STATE,REGION
0,STR-20261115121659-002,arjun.k@yahoo.in,Gold,350.00,INR,350.00,6,Bengaluru,Karnataka,South
1,STR-20261115121701-003,arjun.k@yahoo.in,Gold,400.00,INR,400.00,8,Bengaluru,Karnataka,South
2,STR-20261115121703-004,arjun.k@yahoo.in,Gold,450.00,INR,450.00,8,Bengaluru,Karnataka,South
3,STR-20261115121705-005,arjun.k@yahoo.in,Gold,250.00,INR,250.00,4,Bengaluru,Karnataka,South
4,STR-20261115121707-006,arjun.k@yahoo.in,Gold,300.00,INR,300.00,6,Bengaluru,Karnataka,South
5,STR-20261115121709-007,arjun.k@yahoo.in,Gold,350.00,INR,350.00,6,Bengaluru,Karnataka,South
6,STR-20261115121711-008,arjun.k@yahoo.in,Gold,400.00,INR,400.00,8,Bengaluru,Karnataka,South
7,STR-20261115121713-009,arjun.k@yahoo.in,Gold,450.00,INR,450.00,8,Bengaluru,Karnataka,South
8,STR-20261115121715-010,arjun.k@yahoo.in,Gold,250.00,INR,250.00,4,Bengaluru,Karnataka,South
9,STR-20261115121717-011,arjun.k@yahoo.in,Gold,300.00,INR,300.00,6,Bengaluru,Karnataka,South


## 11. Idempotency test — run the transformations a second time



In [12]:
# Capture Gold totals before rerunning transformations.
before = query("""
    SELECT
        COUNT(*),
        COALESCE(SUM(amount_inr), 0),
        COALESCE(SUM(loyalty_points), 0)
    FROM gold.fct_orders
""")[0]

print("Gold totals before second transformation:", before)


# Rerun the transformations.
run_stream_transformations()


# Capture Gold totals after rerunning transformations.
after = query("""
    SELECT
        COUNT(*),
        COALESCE(SUM(amount_inr), 0),
        COALESCE(SUM(loyalty_points), 0)
    FROM gold.fct_orders
""")[0]

print("Gold totals after second transformation:", after)


# Compare the results.
print("\nIdempotency check:")

print("Row count unchanged:", before[0] == after[0])
print("Revenue unchanged:", float(before[1]) == float(after[1]))
print("Loyalty points unchanged:", before[2] == after[2])


# Assertions: fail if Gold totals change.
assert before[0] == after[0], \
    "Gold row count changed on transformation rerun."

assert float(before[1]) == float(after[1]), \
    "Gold revenue changed on transformation rerun."

assert before[2] == after[2], \
    "Gold loyalty points changed on transformation rerun."


print("\nPASS: rerunning transformations did not double Gold totals.")

Gold totals before second transformation: (33, Decimal('17985.00'), 347)
Stream-aware FreshCart transformations completed.
Gold totals after second transformation: (33, Decimal('17985.00'), 347)

Idempotency check:
Row count unchanged: True
Revenue unchanged: True
Loyalty points unchanged: True

PASS: rerunning transformations did not double Gold totals.


In [13]:
conn.close()
print("Snowflake connection closed.")

Snowflake connection closed.
